# Training application model

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from anomaly_detection.parser.data_parsers import parse_system
from anomaly_detection.transformers import system_transformer

In [ ]:
# Path to the data
project_folder = Path.cwd().parent

output_path = project_folder / "data/training/system"

model_path = project_folder / "src/anomaly_detection/models"

transformer_path = project_folder / "src/anomaly_detection/storage/transformers"

system_evtx_path = project_folder / "data/raw/93_syslog.evtx"

system_evtx_path

# Data preparation

In [ ]:
system_training_records = parse_system(system_evtx_path)

system_training_records[0]

In [ ]:
flat_system_training_record = [{**record['timestamp'], **record['profile'], **record['data']} for record in system_training_records]

flat_system_training_record[0]

In [ ]:
system_training_df = pd.DataFrame(flat_system_training_record)

system_training_df.head(10)

In [ ]:
import dill

with open(transformer_path / "system_transformer.pkl", "wb") as f:
    dill.dump(system_transformer, f)

In [ ]:
X_system = system_transformer.transform(system_training_df)

X_system[0]

In [ ]:
X_system = np.nan_to_num(X_system)

X_system[0]

In [ ]:
np.savetxt(output_path / "system_features.csv", X_system, delimiter=",", fmt='%.18e')

In [ ]:
X_system = np.loadtxt(output_path / "system_features.csv", delimiter=",")

In [ ]:
pd.DataFrame(X_system).describe().T

# Modeling

In [ ]:
from sklearn.ensemble import IsolationForest

system_model = IsolationForest(
    n_estimators=512,
    contamination='auto',
    max_features=0.5,
    max_samples=512,
    n_jobs=-1,
)

system_model.fit(X_system)

In [ ]:
scores = system_model.score_samples(X_system)
predictions = system_model.predict(X_system)

In [ ]:
results_df = system_training_df.copy()

results_df["anomaly_score"] = scores
results_df["prediction"] = predictions

results_df.head()

In [ ]:
from anomaly_detection.utils.score_diagnostics import run_diagnostics

run_diagnostics(system_model, X_system, known_anomaly_mask=None)

In [ ]:
decision_scores = system_model.decision_function(X_system)

new_contamination = np.mean(decision_scores < 0.05)

new_contamination

In [ ]:
anomalies_df = results_df[results_df['prediction'] == -1]

anomalies_df

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(anomalies_df['event_id'].value_counts())

In [ ]:
# Extract samples from each anomalous event ID
def pick_system_samples(n_samples_per_event):
    for event_id, group in anomalies_df.groupby("event_id"):
        print(f"\n{'=' * 60}")
        print(f"EventID {event_id}  --  {len(group)} anomalous rows")
 
        idx_min, idx_max = group.index.min(), group.index.max()
        idx_span = idx_max - idx_min
        print(f"row-index range: {idx_min} -> {idx_max}  (span: {idx_span})")
        print(f"{'=' * 60}")
 
        sample = group.sort_values("anomaly_score").head(n_samples_per_event)
 
        for idx, row in sample.iterrows():
            print(f"\n  row index:        {idx}")
            print(f"  event_record_id:  {row['event_record_id']}")
            print(f"  anomaly_score:    {row['anomaly_score']:.4f}")
 
            for col in [
                "event_id",
                "previous_event_id",
                "event_id_frequency",
                "version",
                "correlation_activity_id",
                "execution_thread_id",
            ]:
                if col in row and pd.notna(row[col]):
                    val = str(row[col])[:300]
                    print(f"  {col:<16}: {val}")
            
            for col in [
                "actor",
                "process_name",
                "session_id",
                "status",
                "old_value",
                "new_value",
                "entity",
                "context",
            ]:
                if col in row and pd.notna(row[col]):
                    val = str(row[col])[:300]
                    print(f"  {col:<16}: {val}")

In [ ]:
from contextlib import redirect_stdout

with open(output_path / 'anomalous_samples.txt', 'w') as f:
    with redirect_stdout(f):
        pick_system_samples(5)

In [ ]:
import joblib

joblib.dump(system_model, model_path / "system_model.joblib")